# Dataset loader
We will use this script to convert image dataset to pkl file format

## KITTI dataset

In [1]:
import os
import pickle
import numpy as np
from PIL import Image
from glob import glob
import matplotlib.pyplot as plt
from tqdm import tqdm

In [4]:
def pickler(DATA_PATH):
    images = sorted(glob(DATA_PATH))
    terrain_patches = []

    for img in tqdm(images):
        img = Image.open(img).convert('L')
        img = img.resize((40, 40))
        terrain_patches.append(np.array(img))

    terrain_patches = np.array(terrain_patches)
    print(len(terrain_patches))

    with open("./vertiencoder/data/train/data_train.pkl", "wb") as f:
        pickle.dump(terrain_patches, f)

In [5]:
DIR_PATH = os.path.abspath('.')
# IMAG_PATH = os.path.join(DIR_PATH, 'dataset', 'RUGD', 'RUGD_frames-with-annotations', 'creek', '*.png')
IMAG_PATH = os.path.join(DIR_PATH, 'dataset', 'test_data1', 'cam', 'rgb', '*.png')

pickler(IMAG_PATH)

100%|██████████| 341/341 [00:02<00:00, 134.27it/s]

341


## Davis Hall dataset

In [132]:
import os
import json
import pickle
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm

In [133]:
# Path to dataset folder
DATASET_PATH = "./dataset/test_data1"

# Load Camera Parameters
with open(os.path.join(DATASET_PATH, "cam_param.json"), "r") as f:
    cam_params = json.load(f)

# Load LiDAR Parameters
with open(os.path.join(DATASET_PATH, "lidar_param.json"), "r") as f:
    lidar_params = json.load(f)

# Load IMU Parameters
with open(os.path.join(DATASET_PATH, "imu_param.json"), "r") as f:
    imu_params = json.load(f)

In [134]:
# Load Motion Data
pose_data = np.loadtxt(os.path.join(DATASET_PATH, "pose.txt"))  # (x, y, z, roll, pitch, yaw)
vel_data = np.loadtxt(os.path.join(DATASET_PATH, "vel.txt"))  # (v, ω)
wheel_speed = np.loadtxt(os.path.join(DATASET_PATH, "wheel_speed.txt"))  # Motor speeds
timestamps = np.loadtxt(os.path.join(DATASET_PATH, "timestamp.txt"))  # Time

# Load RGB Images (Convert to 40x40 Grayscale)
image_paths = sorted(glob(os.path.join(DATASET_PATH, "cam/rgb/*.png")))  # Load RGB images
footprint_images = []
for img_path in tqdm(image_paths):
    img = Image.open(img_path).convert("L")  # Convert to grayscale
    img = img.resize((40, 40))  # Resize to 40x40
    footprint_images.append(np.array(img))

footprint_images = np.array(footprint_images)

100%|██████████| 341/341 [00:02<00:00, 122.81it/s]


In [135]:
len(footprint_images), len(vel_data), len(pose_data)

(341, 341, 341)

In [136]:
pose_data[0]

array([ 6.06286153e-03,  1.49610732e-03,  7.99741317e-03, -4.45195474e-04,
       -3.74356721e-04, -1.65661704e-03,  9.99989867e-01])

In [147]:
# Compute Time Differences
dt_values = np.diff(timestamps, prepend=timestamps[0])

# Compute Pose Differences
pose_diff = np.diff(pose_data, axis=0, prepend=pose_data[0:1, :])

# Create Dataset Dictionary
dataset = {
    "bag_name": ["test_data1"],
    "data": []
}

# Structure Data for Training
num_samples = min(len(footprint_images), len(vel_data), len(pose_data))  # Ensure matching sizes

# for i in range(num_samples):
#     sample = {
#         "cmd_vel": vel_data[i],  # (v, ω)
#         "footprint": footprint_images[i],  # 40x40 grayscale image
#         "pose": pose_data[i],  # (x, y, z, roll, pitch, yaw)
#         "motor_speed": wheel_speed[i] if len(wheel_speed) > i else np.zeros(4),  # If missing, use zeros
#         "dt": np.array(dt_values[i]).reshape((1,)),  # Time difference
#         "pose_diff": pose_diff[i],  # Pose changes
#         "time": np.array(timestamps[i]).reshape((1,))  # Timestamp
#     }
#     dataset["data"].append(sample)

sample = {
    "cmd_vel": vel_data,  # (v, ω)
    "footprint": footprint_images,  # 40x40 grayscale image
    "pose": pose_data,  # (x, y, z, roll, pitch, yaw)
    "motor_speed": wheel_speed,  # If missing, use zeros
    "dt": np.array(dt_values),  # Time difference
    "pose_diff": pose_diff,  # Pose changes
    "time": np.array(timestamps)  # Timestamp
}

dataset["data"].append(sample)

In [148]:
# Save Processed Data
with open("./vertiencoder/data/train/data_train_filtered.pkl", "wb") as f:
    pickle.dump(dataset, f)

print("✅ Dataset saved as data_train_filtered.pkl with", num_samples, "samples.")


✅ Dataset saved as data_train_filtered.pkl with 341 samples.


### Running the script

In [149]:
! python ./vertiencoder/utils/stats.py

Traceback (most recent call last):
  File "c:\Users\future\Dev\Seminar\VertiEncoder\vertiencoder\utils\stats.py", line 75, in <module>
    calculate_stats("vertiencoder/data/train/data_train_filtered.pkl")
  File "c:\Users\future\Dev\Seminar\VertiEncoder\vertiencoder\utils\stats.py", line 41, in calculate_stats
    footprint_pose_list.append(transform(read_patch(f'{root.parents[0]}/{footprint}', pose[2])))
  File "c:\Users\future\Dev\Seminar\VertiEncoder\vertiencoder\utils\helpers.py", line 62, in read_patch
    patch = to_tensor(np.load((patch_name).as_posix())).unsqueeze(0)
AttributeError: 'str' object has no attribute 'as_posix'


### Opening the pickle file

In [140]:
with open('./vertiencoder/data/train/data_train_filtered.pkl', 'rb') as f:
    global content
    content = pickle.load(f)
    print(content.keys())

dict_keys(['bag_name', 'data'])


### Sample data

In [141]:
with open('./dataset/test_data1/data_train.pickle', 'rb') as f:
    content2 = pickle.load(f)

content2.keys()

dict_keys(['bag_name', 'data', 'total_records'])

In [142]:
np.array(content2['data'][0]['pose']).shape

(169, 6)

In [143]:
np.array(content['data'][0]['pose']).shape

(341, 7)

In [144]:
content2['data'][0].keys()

dict_keys(['cmd_vel', 'elevation_map', 'footprint', 'pose', 'motor_speed', 'dt', 'map_offset'])